# Đánh giá khả thi: trợ lý hỏi đáp pháp luật

Bạn là AI Engineer của một công ty phần mềm pháp lý. Product manager muốn thêm tính năng **hỏi đáp pháp luật tiếng
Việt có trích dẫn điều luật** cho khoảng **2 000 người dùng hoạt động, mỗi người khoảng 5 câu hỏi mỗi ngày**. Trước khi
ai viết dòng code nào, bạn cần một bản đánh giá khả thi (`MEMO.md`) có số đo: mỗi request tốn bao nhiêu token, bao nhiêu
tiền, chờ bao lâu, model nào được phép dùng, rủi ro gì.

Notebook này dựng các con số đó trên **dữ liệu thật** (Zalo AI Legal Text Retrieval: 61 425 điều luật, câu hỏi thật của
người dùng) và dùng lại hàm của các bài trước: `cost.py` (Basic 01), `model_card.py` (Basic 02), và hệ số độ trễ bạn đo
ở Intermediate 02. Làm các bài đó trước.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell `explore` đọc dữ liệu thật, gọi hàm của bạn, in bảng. Thêm cell `explore` tuỳ ý.
- Kết quả cuối là `MEMO.md` cạnh notebook. Chấm: `uv run pytest tracks/ai-engineer/modules/00-introduction/exercises/advanced/01-ai-feature-brief -q`

In [ ]:
# explore: gốc repo, dữ liệu, tokenizer, và module của các bài trước
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

MODULE = ROOT / "tracks/ai-engineer/modules/00-introduction"
BASIC = MODULE / "exercises/basic"


def load(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


cost = load(BASIC / "01-inference-cost" / "starter" / "cost.py")
model_card = load(BASIC / "02-model-card" / "starter" / "model_card.py")
ZALO = ensure("zalo-legal")
qwen = Tokenizer.from_file(hf_hub_download("Qwen/Qwen2.5-0.5B-Instruct", "tokenizer.json"))


def encode(text: str) -> list[int]:
    return qwen.encode(text, add_special_tokens=False).ids

## 1. Một request RAG tốn bao nhiêu token?

Mỗi request gồm: system prompt (chỉ thị, cố định khoảng `system_tokens` token), câu hỏi, và `k` điều luật retrieval
tìm được. Điều luật có thể rất dài; một cách kiểm soát là **cắt mỗi điều** ở `max_article_tokens` token.

- `request_tokens(question, articles, encode, system_tokens=150, max_article_tokens=None, k=5) -> int`:
  `system_tokens` + số token câu hỏi + tổng số token của **tối đa `k` điều đầu tiên**, mỗi điều tính
  `min(số token, max_article_tokens)` (`None` là không cắt).
- `overflow_rate(tokens, max_new_tokens, context_window) -> float`: tỉ lệ request mà prompt cộng `max_new_tokens`
  **vượt** context window. Danh sách rỗng thì `ValueError`.

In [ ]:
def request_tokens(question: str, articles: list[str], encode, system_tokens: int = 150,
                   max_article_tokens: int | None = None, k: int = 5) -> int:
    # TODO
    raise NotImplementedError


def overflow_rate(tokens: list[int], max_new_tokens: int, context_window: int) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: 300 câu hỏi ghim (300 query-id đầu của qrels/test.jsonl, như module 13) và điều luật đáp án
qrels: dict[str, list[str]] = {}
with open(ZALO / "qrels" / "test.jsonl", encoding="utf-8") as f:
    for row in map(json.loads, f):
        if row["query-id"] in qrels or len(qrels) < 300:
            qrels.setdefault(row["query-id"], []).append(row["corpus-id"])
with open(ZALO / "queries.jsonl", encoding="utf-8") as f:
    queries = {q["_id"]: q["text"] for q in map(json.loads, f) if q["_id"] in qrels}
corpus, by_law = {}, {}
with open(ZALO / "corpus.jsonl", encoding="utf-8") as f:
    for d in map(json.loads, f):
        corpus[d["_id"]] = d["title"] + "\n" + d["text"]
        by_law.setdefault(d["_id"].rsplit("+", 1)[0], []).append(d["_id"])


def retrieved(qid: str) -> list[str]:
    # Giả lập retrieval top-5: điều đáp án trước, rồi các điều khác của cùng văn bản theo thứ tự trong corpus.
    gold = qrels[qid]
    law = gold[0].rsplit("+", 1)[0]
    return [corpus[i] for i in (gold + [i for i in by_law[law] if i not in gold])[:5]]


q_tok = np.array([len(encode(queries[q])) for q in qrels])
gold_tok = np.array([len(encode(corpus[c])) for q in qrels for c in qrels[q]])
print("câu hỏi: trung vị", np.median(q_tok), "token, lớn nhất", q_tok.max())
print("điều đáp án:", len(gold_tok), "điều, trung vị", np.median(gold_tok), "token, trung bình", round(gold_tok.mean()),
      ", dài hơn 1 500 token:", f"{(gold_tok > 1500).mean():.1%}")

In [ ]:
# explore: phân phối token mỗi request với các mức cắt, và tỉ lệ tràn context 4 096 (mặc định Ollama) khi trả lời 300 token
rows, TOKENS = [], {}
for cap in [None, 1500, 800]:
    toks = [request_tokens(queries[q], retrieved(q), encode, 150, cap, 5) for q in qrels]
    TOKENS[cap] = toks
    rows.append({"cắt mỗi điều": cap or "không", "trung vị": np.median(toks), "p95": np.percentile(toks, 95),
                 "trung bình": round(np.mean(toks), 1), "lớn nhất": max(toks),
                 "tràn 4 096": f"{overflow_rate(toks, 300, 4096):.1%}"})
display(pd.DataFrame(rows))

## 2. Chi phí các phương án

Giá API dưới đây là **giá công bố kiểm ngày 2026-10-30** (mục 5 lý thuyết), USD/MTok. Giá thuê GPU là **giả định**: tự
tìm giá thật ở nơi bạn định thuê và sửa. Số token đếm bằng tokenizer Qwen2.5; tokenizer của nhà cung cấp API khác sẽ cho
số khác (mục 5 lý thuyết) - ghi rõ giả định này trong `MEMO.md`.

In [ ]:
# explore: chi phí tháng cho 2 000 người dùng × 5 câu/ngày với mức cắt bạn chọn
REQUESTS_PER_DAY = 2000 * 5
CAP = 800
N_IN = float(np.mean(TOKENS[CAP]))
N_OUT = 300
OPTIONS = {
    "Claude Haiku 5.5": {"input": 0.10, "output": 0.50, "cached_input": 0.01},
    "Gemini 3.8 Flash (giá tới 2026-12-31)": {"input": 0.75, "output": 3.75, "cached_input": 0.075},
    "Gemini 3.8 Flash (từ 2027-01-01)": {"input": 1.50, "output": 7.50, "cached_input": 0.15},
    "Claude Sonnet 5.5": {"input": 2.0, "output": 10.0, "cached_input": 0.10},
}
cache_hit = 150 / N_IN  # chỉ system prompt lặp lại giữa các request
rows = [{"phương án": name, "USD/tháng": round(cost.monthly_cost(REQUESTS_PER_DAY, N_IN, N_OUT, price), 1),
         "USD/tháng có cache system": round(cost.monthly_cost(REQUESTS_PER_DAY, N_IN, N_OUT, price, cache_hit), 1)}
        for name, price in OPTIONS.items()]
GPU_USD_PER_HOUR = 0.5        # GIẢ ĐỊNH: thuê một GPU 24GB, sửa theo giá thật bạn tìm được
SELF_HOST_TPS = 400           # GIẢ ĐỊNH: tổng token/giây khi phục vụ nhiều request song song
self_host = GPU_USD_PER_HOUR * 24 * 30  # thuê nguyên tháng, bận hay không cũng trả
rows.append({"phương án": "Tự host (giả định GPU 0.5 USD/giờ, 24/7)", "USD/tháng": self_host,
             "USD/tháng có cache system": self_host})
display(pd.DataFrame(rows))
tokens_per_day = REQUESTS_PER_DAY * (N_IN + N_OUT)
print(f"{tokens_per_day / 1e6:.1f} triệu token/ngày; GPU cần bận "
      f"{tokens_per_day / (SELF_HOST_TPS * 86400):.0%} thời gian nếu đạt {SELF_HOST_TPS} token/giây")

## 3. Độ trễ và model

Dán hệ số mô hình độ trễ bạn khớp ở bài Intermediate 02 (`a`, `b`, `c`). Mặc định là số đo **GPU rảnh** trong mục 4 lý
thuyết (prefill khoảng 3 300 token/giây, decode khoảng 74 token/giây với `qwen2.5:3b` trên RTX 3050 Laptop).

In [ ]:
# explore: độ trễ dự đoán cho request trung vị và p95, và giấy phép của các model mở ứng viên
LATENCY = {"a": 0.2, "b": 1 / 3300, "c": 1 / 74}   # thay bằng hệ số của bạn
for label, n_in in [("trung vị", np.median(TOKENS[CAP])), ("p95", np.percentile(TOKENS[CAP], 95))]:
    t = LATENCY["a"] + LATENCY["b"] * n_in + LATENCY["c"] * N_OUT
    print(f"request {label}: {n_in:.0f} token vào, {N_OUT} ra -> khoảng {t:.1f} giây (TTFT {LATENCY['a'] + LATENCY['b'] * n_in:.2f} giây)")

FIX = MODULE / "exercises/basic/02-model-card/tests/fixtures"
cards = []
for d in sorted(FIX.iterdir()):
    s = model_card.card_summary((d / "README.md").read_text(encoding="utf-8"),
                                json.loads((d / "config.json").read_text(encoding="utf-8")))
    cards.append({"model": d.name.replace("__", "/"), "giấy phép": s["license"], "tham số (tỉ)": round(s["params"] / 1e9, 2),
                  "context": s["context_length"],
                  "vừa 4GB (4 bit)": model_card.fits_on_gpu(json.loads((d / "config.json").read_text()), 4, 4, 4096)})
display(pd.DataFrame(cards))

## 4. Viết `MEMO.md`

Mở `MEMO.md` cạnh notebook và điền từng phần bằng số **từ notebook này** (và từ các bài trước), không phải số đoán.
Bản memo dành cho product manager và trưởng nhóm kỹ thuật đọc: ngắn, có bảng, có khuyến nghị rõ ràng và điều kiện để
khuyến nghị đó sai.